# WSJ 2027 - Gruppindelning Ledare

Assign interviewed leaders into groups of 4 based on:
1. **Pair/quartet wishes** — honor pre-formed groups
2. **Geographic proximity** — Bostadsort from interview form

Data source: `Formulär ledarintervju.xlsx`

In [1]:
# Prerequisites
import subprocess, sys
for pkg in ['openpyxl', 'geopy']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
        print(f'Installed {pkg}')
print('Prerequisites OK')

Prerequisites OK


In [2]:
import sys
sys.path.insert(0, '/config/notebooks/wsj27')
import wsj27_utils as u
import pandas as pd
import numpy as np
from difflib import SequenceMatcher

# Load Excel
EXCEL_PATH = '/config/notebooks/wsj27/input/Formulär ledarintervju (1).xlsx'
df_raw = pd.read_excel(EXCEL_PATH)
print(f'Loaded {len(df_raw)} leaders from Excel')
print(f'\nKön: {df_raw["Kön"].value_counts().to_dict()}')
print(f'Bedömning: {df_raw["Sammanfattande bedömning efter referenstagning"].value_counts().to_dict()}')

# Build clean DataFrame
df = pd.DataFrame({
    'name': df_raw['Namn1'],
    'sex': df_raw['Kön'],
    'birth_year': df_raw['Födelseår'].astype(str).str.extract(r'(\d{4})')[0].astype(float),
    'bostadsort': df_raw['Bostadsort'],
    'kar': df_raw['Medlem i scoutkår/förbund'],
    'recommendation': df_raw['Sammanfattande bedömning efter referenstagning'],
    'pair_raw': df_raw['Har anmält sig i par eller kvartett. Ange person ett här. Namn och kår'],
    'q2_raw': df_raw['Har anmält sig i kvartett. Ange person två här. Namn och kår'],
    'q3_raw': df_raw['Har anmält sig i kvartett. Ange person tre här. Namn och kår'],
    'travel_pref': df_raw['Vilken lägerupplevelse önskas'],
}).reset_index(drop=True)

# Map travel preference to rundresa/direktresa
travel_map = {'Med rundresa': 'rundresa', 'Utan rundresa': 'direktresa',
              'Kan tänka mig vilket som': 'flexibel'}
df['travel'] = df['travel_pref'].map(travel_map).fillna('flexibel')

df['age'] = 2027 - df['birth_year']

# Split: only "Rekommenderar" enters grouping; rest are listed in summary
df_excluded = df[df['recommendation'] != 'Rekommenderar'].copy().reset_index(drop=True)
df = df[df['recommendation'] == 'Rekommenderar'].copy().reset_index(drop=True)
print(f'\nIngår i gruppindelning: {len(df)} ledare')
print(f'Exkluderade (ej rekommenderade): {len(df_excluded)} ledare')

print(f'\nAge range: {df["age"].min():.0f}-{df["age"].max():.0f}')
print(f'\nRestyp:')
print(df['travel'].value_counts().to_string())
print(f'\nBostadsort samples:')
print(df[['name', 'bostadsort', 'travel']].head(10).to_string(index=False))

Loaded 249 leaders from Excel

Kön: {'Kvinna': 129, 'Man': 120}
Bedömning: {'Rekommenderar': 234, 'Osäker - komplettering behövs': 11, 'Rekommenderar ej': 4}

Ingår i gruppindelning: 234 ledare
Exkluderade (ej rekommenderade): 15 ledare

Age range: 19-69

Restyp:
travel
rundresa      146
direktresa     55
flexibel       33

Bostadsort samples:
                     name      bostadsort     travel
        Malin Otter Fröjd      Arvidsjaur   flexibel
             Jonas Astros Viksjö Järfälla   rundresa
Maria Hagsäter Lagerkvist             Ale direktresa
         Susanna Rönnback          Gnesta direktresa
  Johanna Perryd Mattsson        Göteborg   rundresa
           Sara Bengtsson            Lund   flexibel
            Linnea Irwert        Göteborg direktresa
          Caroline Meijer        Jonstorp   rundresa
          Florenz Sörling        Karlstad direktresa
              Simon Jovin           Malmö   rundresa


In [3]:
# Manuella overrides för bostadsorter som inte kan geocodas automatiskt.
# Värdet 'okänt' betyder Stockholm. Ange annars ett stadsnamn som ska
# geocodas mot (t.ex. 'Mölnlycke', 'Vejbystrand', 'Örebro').
# Cellen failar om angiven stad inte kan geocodas.
MANUAL_OVERRIDES = {
    'Falun men flyttar till Örebro till hösten': 'Örebro',
    'Mölnycke': 'Mölnlycke',
    'Olofstorp utanför Göteborg': 'Olofstorp',
    'Segersäng, söder om Stockholm': 'Segersäng',
    'Umeå/Nyköping': 'Nyköping',
    'Veijbystrand': 'Vejbystrand',
}

In [4]:
# Geocode Bostadsort -> lat/lng
u.geocode_places(df, place_column='bostadsort', manual_overrides=MANUAL_OVERRIDES)

# Hilbert sort for geographic ordering
df = u.add_hilbert_index(df)
print(f'\nSorted {len(df)} leaders by geographic proximity')

Resolving 6 manuella overrides...
  'Falun men flyttar till Örebro till hösten' -> Örebro (59.2747, 15.2151)
  'Mölnycke' -> Mölnlycke (57.6584, 12.1136)
  'Olofstorp utanför Göteborg' -> Olofstorp (57.7986, 12.1686)
  'Segersäng, söder om Stockholm' -> Segersäng (59.0272, 17.9366)
  'Umeå/Nyköping' -> Nyköping (58.7545, 17.0121)
  'Veijbystrand' -> Vejbystrand (56.3142, 12.7656)

With coordinates: 234
Without coordinates (Sweden centroid): 0

Sorted 234 leaders by geographic proximity


In [5]:
# Manuella overrides för par-/kvartettönskemål som inte kan matchas automatiskt.
# Nyckel = det extraherade namnet från Excel-fältet (visas i unmatched-listan).
# Värde:
#   None        -> skippa (den andra personen har inte sökt)
#   'Förnamn Efternamn' -> matcha mot den personen i datasetet
# Cellen failar om angivet namn inte finns i datasetet.
WISH_OVERRIDES = {
    'Timmie Nilsson Sjöbo': 'Timmie Nilsson',
    'Celia Bjerstedt': None,
    'Erik Olsson': None,
    'Nej': None,
    'Mikael Wahlborg': None,
    'Ann-Louise Ekestubbe Masthugget Majornas': 'Ann-Louise Ekestubbe',
    'Martin Kropp': None,
    'Ja': None,
    'Önskat': None,
    'Vill är': None,
    'Har anmält sig individuellt': None,
    'Per Harding': None,
    'Emil Stridfeldt Viksjö': 'Emil Stridfeldt',
    'Helena Zenzén': None,
    'Magnus Hillerdal Östhammars sjöscoutkår': 'Magnus Hillerdal',
    'Ingrid Ahnlund Rode': None,
    'Joakim Huss': None,
}

In [6]:
# Parse pair/quartet wishes
# Build name lookup for fuzzy matching (active leaders only)
name_lookup = {}
for idx, row in df.iterrows():
    norm = row['name'].lower().strip()
    name_lookup.setdefault(norm, []).append(idx)
    parts = norm.split()
    if len(parts) >= 2:
        short = parts[0] + ' ' + parts[-1]
        name_lookup.setdefault(short, []).append(idx)

# Lookup for excluded leaders — to detect wishes that point to someone we're not grouping
excluded_lookup = set()
for _, row in df_excluded.iterrows():
    norm = row['name'].lower().strip()
    excluded_lookup.add(norm)
    parts = norm.split()
    if len(parts) >= 2:
        excluded_lookup.add(parts[0] + ' ' + parts[-1])

def extract_name(text):
    """Extract person name from free-text wish field."""
    import re
    if not isinstance(text, str) or not text.strip():
        return None
    text = text.strip()
    # Skip non-person entries
    if text.lower() in ('ensam', 'nej', 'kvartet', 'inga', 'ingen'):
        return None
    # Remove leading qualifiers
    prefixes = [
        'Önskemål (inte anmält sig tillsammans) ',
        'Gärna tillsammans med ',
        'Önskar eventuellt ',
        'Han söker som par. ',
    ]
    for prefix in prefixes:
        if text.startswith(prefix):
            text = text[len(prefix):]
    # Split on separators: comma, period, parenthesis, "från", kår-words
    name_part = re.split(r'[,.(]|\sfrån\s|\s(?:[Ss]coutkår|Scoutkåren)', text)[0].strip()
    # Remove trailing kår-like words that slipped through
    name_part = re.sub(r'\s+(scoutkår|Scoutkåren|scoutkåren).*$', '', name_part, flags=re.IGNORECASE)
    return name_part if name_part else None

def match_name(text):
    """Try to match a free-text name reference.
    Returns (status, idx_or_None, name_part) where status is
    'matched', 'excluded', or 'unmatched'."""
    name_part = extract_name(text)
    if not name_part:
        return 'unmatched', None, name_part
    norm = name_part.lower().strip()
    if norm in name_lookup:
        return 'matched', name_lookup[norm][0], name_part
    if norm in excluded_lookup:
        return 'excluded', None, name_part
    # Fuzzy match against active df
    best_score, best_idx = 0, None
    for idx2, row2 in df.iterrows():
        s1 = SequenceMatcher(None, norm, row2['name'].lower()).ratio()
        parts = row2['name'].lower().split()
        s2 = SequenceMatcher(None, norm, parts[0] + ' ' + parts[-1]).ratio() if len(parts) >= 2 else 0
        score = max(s1, s2)
        if score > best_score:
            best_score = score
            best_idx = idx2
    if best_score >= 0.85:
        return 'matched', best_idx, name_part
    return 'unmatched', None, name_part

def apply_wish_override(name_part):
    """Resolve an unmatched name through WISH_OVERRIDES.
    Returns ('skip', None), ('match', idx), ('excluded', None), or ('absent', None)."""
    if name_part not in WISH_OVERRIDES:
        return 'absent', None
    target = WISH_OVERRIDES[name_part]
    if target is None:
        return 'skip', None
    norm_target = target.lower().strip()
    if norm_target in name_lookup:
        return 'match', name_lookup[norm_target][0]
    if norm_target in excluded_lookup:
        return 'excluded', None
    raise ValueError(
        f"WISH_OVERRIDES: '{name_part}' -> {target!r} matchar ingen i datasetet. "
        f"Kontrollera stavningen."
    )

# Parse all wishes
wish_groups = []
processed = set()
matched_log = []
unmatched_log = []
override_log = []
excluded_log = []

for idx, row in df.iterrows():
    if idx in processed:
        continue
    results = []
    for col in ['pair_raw', 'q2_raw', 'q3_raw']:
        raw = row[col]
        status, matched_idx, name_part = match_name(raw)
        if name_part is None:
            continue
        if status == 'matched':
            results.append(matched_idx)
            matched_log.append(f"  {df.at[idx, 'name']}: '{name_part}' -> {df.at[matched_idx, 'name']}")
            continue
        if status == 'excluded':
            excluded_log.append(f"  {df.at[idx, 'name']}: '{name_part}' (önskad person ej rekommenderad)")
            continue
        # status == 'unmatched' — consult WISH_OVERRIDES
        ov_status, override_idx = apply_wish_override(name_part)
        if ov_status == 'skip':
            override_log.append(f"  {df.at[idx, 'name']}: '{name_part}' (skippad via WISH_OVERRIDES)")
        elif ov_status == 'match':
            results.append(override_idx)
            override_log.append(f"  {df.at[idx, 'name']}: '{name_part}' -> {df.at[override_idx, 'name']} (override)")
        elif ov_status == 'excluded':
            excluded_log.append(f"  {df.at[idx, 'name']}: '{name_part}' -> {WISH_OVERRIDES[name_part]} (ej rekommenderad)")
        else:
            unmatched_log.append(f"  {df.at[idx, 'name']}: '{name_part}' (not in dataset)")
    
    partners = set(results)
    if not partners:
        continue
    # Build group and merge overlapping
    group = {idx} | partners
    merged = set()
    for gi, wg in enumerate(wish_groups):
        if group & wg:
            merged.add(gi)
            group |= wg
    wish_groups = [wg for gi, wg in enumerate(wish_groups) if gi not in merged]
    wish_groups.append(group)
    processed |= group

print('=== Matched Wishes ===')
for line in matched_log:
    print(line)

if override_log:
    print(f'\n=== WISH_OVERRIDES tillämpade ===')
    for line in override_log:
        print(line)

if excluded_log:
    print(f'\n=== Önskemål mot exkluderade ledare ===')
    for line in excluded_log:
        print(line)

print(f'\n=== Unmatched (person saknas i WISH_OVERRIDES) ===')
for line in unmatched_log:
    print(line)
if unmatched_log:
    print('\nLägg till ovanstående i WISH_OVERRIDES (None för att skippa, eller ett namn att matcha mot).')

print(f'\n=== Wish Groups ===')
for wg in sorted(wish_groups, key=lambda s: -len(s)):
    names = [df.at[i, 'name'] for i in sorted(wg)]
    print(f'  Group of {len(wg)}: {", ".join(names)}')

singles = set(df.index) - processed
print(f'\nWith wishes: {len(processed)}')
print(f'Singles (no wish): {len(singles)}')

# Store for report cell
_unmatched_log = unmatched_log
_override_log = override_log
_excluded_log = excluded_log

=== Matched Wishes ===
  Linus Håbecker: 'Sofia Broman' -> Sofia Broman
  Linus Håbecker: 'Adrian Cammelli Hansson' -> Adrian Cammeli Hansson
  Linus Håbecker: 'Jenny Jönsson' -> Jenny Jönsson
  Vilgot Asplund: 'My Gärdebring' -> My Gärdebring
  Jeanette Nettan Arksund: 'Johannes Fridlund Wä' -> Johannes Fridlund
  My Forssander: 'Anna Lundberg' -> Anna Lundberg
  Hilma Cederlund: 'Signe Winberg' -> Signe Wiberg
  Eva Svanström: 'Erik Svanström' -> Erik Svanström
  Anne Boklund: 'Jeanette Larsson' -> Jeanette Larsson
  Anne Boklund: 'Mathias Johansson' -> Mathias Johansson
  Anne Boklund: 'Johan Eriksson' -> Johan Eriksson
  Ebba Källström: 'Joel Kruse' -> Joel Kruse
  Theodor Söderberg: 'Julia Andersson' -> Julia Strömberg Andersson
  Cecilia Formare: 'Johan Wiklander' -> Johan Wiklander
  Cecilia Formare: 'Edvin Palmgren' -> Edvin Palmgren
  Cecilia Formare: 'Elin Jungevi' -> Elin Jungevi
  Rasmus Lindberg: 'Martin Nyberg' -> Martin Nyberg
  Mattias Persson: 'Carola Bengtsson' -> Car

In [7]:
GROUP_SIZE = 4

# Step 1: Assign flexible leaders to rundresa/direktresa
n_rund_fixed = len(df[df['travel'] == 'rundresa'])
n_dir_fixed = len(df[df['travel'] == 'direktresa'])
n_flex = len(df[df['travel'] == 'flexibel'])
print(f'Fasta: {n_rund_fixed} rundresa, {n_dir_fixed} direktresa, {n_flex} flexibla')

# Find optimal split: minimize total remainder across both tracks
best_split = 0
best_remainder = 999
for flex_rund in range(n_flex + 1):
    r = n_rund_fixed + flex_rund
    d = n_dir_fixed + (n_flex - flex_rund)
    remainder = (r % GROUP_SIZE) + (d % GROUP_SIZE)
    # Prefer splits where rundresa remainder is 0 (full groups)
    if remainder < best_remainder or (remainder == best_remainder and r % GROUP_SIZE == 0):
        best_remainder = remainder
        best_split = flex_rund

# Sort flexibles by Hilbert, assign first N to rundresa
flexibles = sorted(df[df['travel'] == 'flexibel'].index.tolist(),
                   key=lambda i: df.at[i, 'hilbert'])
for i, idx in enumerate(flexibles):
    df.at[idx, 'travel'] = 'rundresa' if i < best_split else 'direktresa'

n_rund = len(df[df['travel'] == 'rundresa'])
n_dir = len(df[df['travel'] == 'direktresa'])
print(f'\nEfter fördelning:')
print(f'  Rundresa:   {n_rund} ({n_rund_fixed} fast + {n_rund - n_rund_fixed} flexibla) -> {n_rund // GROUP_SIZE} grupper + {n_rund % GROUP_SIZE}')
print(f'  Direktresa: {n_dir} ({n_dir_fixed} fast + {n_dir - n_dir_fixed} flexibla) -> {n_dir // GROUP_SIZE} grupper + {n_dir % GROUP_SIZE}')

# Step 2: Group within each travel type
n = len(df)
group_of = np.full(n, -1, dtype=int)
member_source = {}
next_group = 0

for travel_type in ['rundresa', 'direktresa']:
    df_travel = df[df['travel'] == travel_type]
    travel_idx = set(df_travel.index)
    print(f'\n{"=" * 50}')
    print(f'=== {travel_type.upper()} ({len(df_travel)} ledare) ===')
    print(f'{"=" * 50}')
    
    # Filter wish groups to this travel type (keep pairs/groups where >=2 in same track)
    travel_wishes = []
    for wg in wish_groups:
        wg_in_travel = wg & travel_idx
        if len(wg_in_travel) >= 2:
            travel_wishes.append(wg_in_travel)
    
    # Phase 1: Quartets
    print(f'\n--- Kvartetter ---')
    quartets = [wg for wg in travel_wishes if len(wg) == GROUP_SIZE]
    for wg in quartets:
        for idx in wg:
            group_of[idx] = next_group
            member_source[idx] = 'önskemål'
        names = [df.at[i, 'name'] for i in sorted(wg)]
        print(f'  Grupp {next_group + 1}: {", ".join(names)}')
        next_group += 1
    if not quartets:
        print('  (inga)')
    
    # Phase 2: Pairs + nearest singles
    print(f'\n--- Par + geografisk fyllnad ---')
    pairs = [wg for wg in travel_wishes if len(wg) == 2]
    travel_processed = set()
    for wg in travel_wishes:
        travel_processed |= wg
    unassigned = sorted(
        [i for i in travel_idx if group_of[i] == -1 and i not in travel_processed],
        key=lambda i: df.at[i, 'hilbert'])
    
    for wg in pairs:
        pl = sorted(wg)
        plat = np.mean([df.at[i, 'lat'] for i in pl])
        plng = np.mean([df.at[i, 'lng'] for i in pl])
        dists = sorted([(i, u.haversine_km(df.at[i,'lat'], df.at[i,'lng'], plat, plng))
                        for i in unassigned], key=lambda x: x[1])
        fill = [d[0] for d in dists[:GROUP_SIZE - len(wg)]]
        for idx in list(wg):
            group_of[idx] = next_group
            member_source[idx] = 'önskemål'
        for idx in fill:
            group_of[idx] = next_group
            member_source[idx] = 'extra'
            unassigned.remove(idx)
        names = [df.at[i, 'name'] for i in sorted(list(wg) + fill)]
        print(f'  Grupp {next_group + 1}: {", ".join(names)}')
        next_group += 1
    if not pairs:
        print('  (inga)')
    
    # Phase 2b: Other wish groups (3, 5+)
    others = [wg for wg in travel_wishes if len(wg) != 2 and len(wg) != GROUP_SIZE]
    if others:
        print(f'\n--- Övriga önskemålsgrupper ---')
        for wg in others:
            wl = sorted(wg)
            if len(wg) < GROUP_SIZE:
                cl = np.mean([df.at[i, 'lat'] for i in wl])
                cn = np.mean([df.at[i, 'lng'] for i in wl])
                dists = sorted([(i, u.haversine_km(df.at[i,'lat'], df.at[i,'lng'], cl, cn))
                                for i in unassigned], key=lambda x: x[1])
                fill = [d[0] for d in dists[:GROUP_SIZE - len(wg)]]
                for idx in wl:
                    group_of[idx] = next_group
                    member_source[idx] = 'önskemål'
                for idx in fill:
                    group_of[idx] = next_group
                    member_source[idx] = 'extra'
                    unassigned.remove(idx)
                names = [df.at[i, 'name'] for i in sorted(wl + fill)]
                print(f'  Grupp {next_group + 1}: {", ".join(names)}')
                next_group += 1
            else:
                ws = sorted(wl, key=lambda i: df.at[i, 'hilbert'])
                for cs in range(0, len(ws), GROUP_SIZE):
                    chunk = ws[cs:cs + GROUP_SIZE]
                    for idx in chunk:
                        group_of[idx] = next_group
                        member_source[idx] = 'önskemål'
                    names = [df.at[i, 'name'] for i in chunk]
                    print(f'  Grupp {next_group + 1}: {", ".join(names)}')
                    next_group += 1
    
    # Phase 3: Remaining by Hilbert
    remaining = sorted([i for i in travel_idx if group_of[i] == -1],
                       key=lambda i: df.at[i, 'hilbert'])
    if remaining:
        print(f'\n--- Geografisk indelning ({len(remaining)} kvar) ---')
        for cs in range(0, len(remaining), GROUP_SIZE):
            chunk = remaining[cs:cs + GROUP_SIZE]
            for idx in chunk:
                group_of[idx] = next_group
                member_source[idx] = 'geo'
            names = [df.at[i, 'name'] for i in chunk]
            print(f'  Grupp {next_group + 1}: {", ".join(names)}')
            next_group += 1

total_groups = next_group
df['group'] = group_of
print(f'\n=== RESULTAT: {total_groups} grupper totalt ===')

Fasta: 146 rundresa, 55 direktresa, 33 flexibla

Efter fördelning:
  Rundresa:   176 (146 fast + 30 flexibla) -> 44 grupper + 0
  Direktresa: 58 (55 fast + 3 flexibla) -> 14 grupper + 2

=== RUNDRESA (176 ledare) ===

--- Kvartetter ---
  Grupp 1: Linus Håbecker, Jenny Jönsson, Adrian Cammeli Hansson, Sofia Broman
  Grupp 2: Anne Boklund, Jeanette Larsson, Mathias Johansson, Johan Eriksson
  Grupp 3: Cecilia Formare, Elin Jungevi, Johan Wiklander, Edvin Palmgren
  Grupp 4: Lydia Dahlberg, Nils Lagnemo, Annika Karlsson, Mats Wahlborg
  Grupp 5: Emil Sone, Linnea Neiderud, Marijn van der Sluijs, Klara Lekås
  Grupp 6: Fredrik Berg, Markus Sjöqvist, Helena Paues, Andreas Söderlind
  Grupp 7: Niklas Åhman, Mikael Sjögren, Sara Eriksson, Johan Fagerlund
  Grupp 8: Nils Mogensen, Maria Ekblom, Sven Westergren, Christian Demandt
  Grupp 9: Manuel Lindström, Anna-Karin Björkman, Jonas Mixter, Jessica Östborg

--- Par + geografisk fyllnad ---
  Grupp 10: Bengt Nilsson, Sara Bengtsson, Erik Ande

In [8]:
# Group overview
for travel_type in ['rundresa', 'direktresa']:
    grps = sorted(df[df['travel'] == travel_type]['group'].unique())
    print(f'\n=== {travel_type.upper()} ({len(grps)} grupper) ===')
    print(f'{"Grupp":>5} {"Storlek":>7} {"M/K":>5} {"Ålder":>12} {"AvståndKm":>10}  Medlemmar')
    print('-' * 110)
    
    all_dists = []
    for g in grps:
        members = df[df['group'] == g]
        size = len(members)
        sex_c = members['sex'].value_counts()
        m_k = f"{sex_c.get('Man', 0)}/{sex_c.get('Kvinna', 0)}"
        ages = members['age'].values
        age_str = f"{int(min(ages))}-{int(max(ages))}" if len(ages) > 1 else f"{int(ages[0])}"
        
        clat, clng = members['lat'].mean(), members['lng'].mean()
        avg_dist = np.mean([u.haversine_km(r['lat'], r['lng'], clat, clng)
                            for _, r in members.iterrows()])
        all_dists.append(avg_dist)
        
        # Annotate names: * = geographic fill, ~ = was flexible
        name_parts = []
        for idx, r in members.iterrows():
            src = member_source.get(idx, '?')
            suffix = ''
            if src == 'extra':
                suffix += ' *'
            if r['travel_pref'] == 'Kan tänka mig vilket som':
                suffix += ' ~'
            name_parts.append(f"{r['name']}{suffix}")
        names = ', '.join(name_parts)
        
        sources = [member_source.get(i, '?') for i in members.index]
        n_wish = sum(1 for s in sources if s == 'önskemål')
        n_extra = sum(1 for s in sources if s == 'extra')
        tag = ''
        if n_wish > 0:
            tag = f' [önskemål:{n_wish}'
            if n_extra > 0:
                tag += f'+extra:{n_extra}'
            tag += ']'
        
        print(f'{g + 1:>5} {size:>7} {m_k:>5} {age_str:>12} {avg_dist:>9.0f}  {names}{tag}')
    
    if all_dists:
        print('-' * 110)
        print(f'Avg geographic spread: {np.mean(all_dists):.0f} km')

print(f'\n* = tillagd geografiskt (ej del av önskemål)')
print(f'~ = flexibel ("Kan tänka mig vilket som")')

# Report: unresolved wishes (person not yet in dataset)
if _unmatched_log:
    print(f'\n=== Ej matchade önskemål (person saknas i data) ===')
    print('Dessa personer har nämnts som par/kvartett-önskemål men finns inte i ledarintervjufilen.')
    print('När en ny version av Excel-filen inkluderar dem kommer de matchas automatiskt.\n')
    for line in _unmatched_log:
        print(line)
    print(f'\nTotalt: {len(_unmatched_log)} önskemål pekar på personer utanför datasetet')

# Report: leaders excluded from grouping (not "Rekommenderar")
if len(df_excluded) > 0:
    print(f'\n=== EXKLUDERADE LEDARE ({len(df_excluded)}) ===')
    print('Dessa ledare ingår inte i gruppindelningen.\n')
    by_rec = df_excluded.groupby('recommendation', sort=False)
    for rec, sub in by_rec:
        print(f'  {rec} ({len(sub)}):')
        for _, r in sub.sort_values('name').iterrows():
            kar = r['kar'] if pd.notna(r['kar']) else ''
            print(f'    - {r["name"]} ({r["bostadsort"]}, {kar})')


=== RUNDRESA (44 grupper) ===
Grupp Storlek   M/K        Ålder  AvståndKm  Medlemmar
--------------------------------------------------------------------------------------------------------------
    1       4   2/2        22-35       448  Linus Håbecker, Jenny Jönsson, Adrian Cammeli Hansson, Sofia Broman [önskemål:4]
    2       4   2/2        34-55       210  Anne Boklund ~, Jeanette Larsson ~, Mathias Johansson ~, Johan Eriksson ~ [önskemål:4]
    3       4   2/2        24-50       197  Cecilia Formare, Elin Jungevi, Johan Wiklander, Edvin Palmgren [önskemål:4]
    4       4   2/2        22-59        43  Lydia Dahlberg, Nils Lagnemo, Annika Karlsson, Mats Wahlborg [önskemål:4]
    5       4   2/2        24-49        21  Emil Sone, Linnea Neiderud, Marijn van der Sluijs, Klara Lekås [önskemål:4]
    6       4   3/1        48-55         6  Fredrik Berg, Markus Sjöqvist, Helena Paues, Andreas Söderlind [önskemål:4]
    7       4   3/1        45-58        14  Niklas Åhman, Mikael Sjög

In [9]:
# Export CSV
OUTPUT_DIR = '/config/notebooks/wsj27/output'

df_export = df[['group', 'name', 'sex', 'age', 'bostadsort', 'kar',
                'travel', 'travel_pref', 'recommendation', 'lat', 'lng']].copy()
df_export['group'] = df_export['group'] + 1  # 1-indexed
df_export = df_export.sort_values(['travel', 'group', 'name']).reset_index(drop=True)

csv_path = f'{OUTPUT_DIR}/wsj27_ledare_grupper.csv'
df_export.to_csv(csv_path, index=False, encoding='utf-8-sig')
print(f'Saved {len(df_export)} leaders to {csv_path}')

# Export JSON
import json as json_mod
group_summary = []
for g in range(total_groups):
    members = df[df['group'] == g]
    travel = members['travel'].iloc[0]
    clat, clng = float(members['lat'].mean()), float(members['lng'].mean())
    group_summary.append({
        'group': g + 1,
        'travel': travel,
        'size': len(members),
        'centroid': {'lat': round(clat, 4), 'lng': round(clng, 4)},
        'members': [{'name': r['name'], 'bostadsort': r['bostadsort'],
                     'kar': str(r['kar']), 'age': int(r['age']),
                     'travel_pref': r['travel_pref']}
                    for _, r in members.iterrows()]
    })

json_path = f'{OUTPUT_DIR}/wsj27_ledare_grupper.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json_mod.dump(group_summary, f, ensure_ascii=False, indent=2)
print(f'Saved group summary to {json_path}')

Saved 234 leaders to /config/notebooks/wsj27/wsj27_ledare_grupper.csv
Saved group summary to /config/notebooks/wsj27/wsj27_ledare_grupper.json


In [10]:
# Generate interactive map
# Prepare df_sorted in the format expected by generate_group_map_html
df_map_input = df[['name', 'kar', 'age', 'lat', 'lng', 'group']].copy()
df_map_input['kar'] = df_map_input['kar'].fillna('')

map_path = f'{OUTPUT_DIR}/wsj_ledare_karta.html'
u.generate_group_map_html(df_map_input, total_groups, map_path,
                          title='WSJ 2027 Ledare', show_group_arcs=True)

print(f'\nAll outputs:')
print(f'  CSV:  {csv_path}')
print(f'  JSON: {json_path}')
print(f'  Map:  {map_path}')

Group arcs: 349 connections across 59 groups
Saved group map: /config/notebooks/wsj27/wsj_ledare_karta.html

All outputs:
  CSV:  /config/notebooks/wsj27/wsj27_ledare_grupper.csv
  JSON: /config/notebooks/wsj27/wsj27_ledare_grupper.json
  Map:  /config/notebooks/wsj27/wsj_ledare_karta.html
